In [ ]:
# ==========================================
# Built-in Python Modules
# ==========================================

import sys   # Access Python interpreter information (version, path, arguments, etc.)
import os    # Interact with the operating system (files, folders, environment variables)


# ==========================================
# LLM (Ollama) Integration
# ==========================================

from langchain_ollama import ChatOllama
# Connect and chat with local Ollama models (Llama, Mistral, Gemma, etc.)


# ==========================================
# External Tools
# ==========================================

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
# Search and retrieve information from Wikipedia

from langchain_community.tools import DuckDuckGoSearchRun
# Search the web using DuckDuckGo


# ==========================================
# Custom Tool Creation
# ==========================================

from langchain_core.tools import tool
# Convert Python functions into LangChain tools using @tool


# ==========================================
# Prompt Templates
# ==========================================

from langchain_core.prompts import ChatPromptTemplate
# Create reusable prompts with dynamic variables


# ==========================================
# ReAct Agent ## use lang-graph
# ==========================================
from langchain.agents import create_agent
#from langgraph.prebuilt import create_react_agent
# Create a ReAct (Reason + Act) agent that can use tools


# ==========================================
# Conversation Memory
# ==========================================

from langgraph.checkpoint.memory import InMemorySaver
# Store conversation history in memory for continuous chatting




C:\Users\Administrator\AppData\Local\Temp\ipykernel_20812\2228350874.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [2]:
# ChatOllama      ← LangChain
# Wikipedia Tool  ← LangChain
# DuckDuckGo Tool ← LangChain
# Prompt          ← LangChain

# create_react_agent ← LangGraph
# InMemorySaver      ← LangGraph
# thread_id          ← LangGraph

In [3]:
print(sys.version)
print(os.getcwd())

3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
d:\github\Learning\RAG\notebook


In [ ]:
# ==========================================
# 1. LLM Setup
# ==========================================
print("🔄 Initializing Ollama (Llama 3.1)...")
llm = ChatOllama(
    model="llama3.1:8b",
    base_url="http://localhost:11434",
    temperature=0.3,
    timeout=60  
)
# ==========================================
# 2. Tools Configuration
# ==========================================
print("📦 Loading search and math tools...")
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
search_tool = DuckDuckGoSearchRun()

@tool
def calculator(expression: str) -> str:
    """Use this tool for math calculations like 2+2, 10*5, etc."""
    try:
        allowed_chars = "0123456789+-*/(). "
        if not all(char in allowed_chars for char in expression):
            return "Error: Invalid characters in math expression."
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

tools = [wiki, search_tool, calculator]
# ==========================================
# 3. Create Modern ReAct Agent with Checkpointer
# ==========================================
print("🤖 Compiling ReAct Agent Graph...")

# Using InMemorySaver handles conversation thread states automatically
memory = InMemorySaver()

# Create a proper prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a helpful assistant. Think step-by-step. "
     "Use your search, wikipedia, and calculator tools whenever you need external or precise information."),
    ("placeholder", "{messages}")
])

# Use create_react_agent with the prompt parameter instead of system_prompt
agent = crea(
    model=llm,
    tools=tools,
    prompt=prompt,  # ✅ Use 'prompt' instead of 'system_prompt'
    checkpointer=memory,
    debug=False  # Toggle to True if you want to inspect low-level LangGraph execution traces
)

🔄 Initializing Ollama (Llama 3.1)...
📦 Loading search and math tools...
🤖 Compiling ReAct Agent Graph...


C:\Users\Administrator\AppData\Local\Temp\ipykernel_20812\3207291999.py:47: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [5]:
UserID = "user1"

config = {
    "configurable": {
        "thread_id": UserID
    }
}

# ==========================================
# 4. Interactive Jupyter Function
# ==========================================
def ask_agent(user_input: str):
    """Call this function inside any Jupyter cell to talk to the agent."""
    if not user_input.strip():
        return
        
    print(f"You: {user_input}")
    print("Thinking...")
    
    try:
        # Pass the message payload structure expected by create_react_agent
        result = agent.invoke(
            {"messages": [{"role": "user", "content": user_input}]}, 
            config=config
        )
        
        # Pull the absolute last response generated in the message graph state
        ai_response = result["messages"][-1].content
        #ai_response = result["messages"]
        print(f"\nAI: {ai_response}\n")
        #print(f"\nMemory: {memory}\n")
    except Exception as e:
        print(f"\nExecution Error: {e}\n")

print("\n🚀 Setup Complete! You can now converse with the agent using the ask_agent() function.")


🚀 Setup Complete! You can now converse with the agent using the ask_agent() function.


In [8]:
ask_agent("Who was Nikola Tesla?")

You: Who was Nikola Tesla?
Thinking...

Execution Error: Found AIMessages with tool_calls that do not have a corresponding ToolMessage. Here are the first few of those tool calls: [{'name': 'wikipedia', 'args': {'query': 'Nikola Tesla'}, 'id': '97e6bd26-e9fa-47aa-97ef-fa7d4c07a53c', 'type': 'tool_call'}].

Every tool call (LLM requesting to call a tool) in the message history MUST have a corresponding ToolMessage (result of a tool invocation to return to the LLM) - this is required by most LLM providers.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CHAT_HISTORY

